In [2]:
import pandas as pd
import numpy as np 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [10]:
customers=pd.read_csv('Telecom_Customer_Churn.csv')
customers.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [12]:
print(customers
      ['customerID'].duplicated().sum())

0


In [14]:
customers=customers.drop_duplicates(subset='customerID').copy()
text_columns=customers.select_dtypes(include=['object','string']).columns
for col in text_columns:
    customers[col]=customers[col].astype('string').str.strip().replace('',pd.NA)

customers['gender']=customers['gender'].str.lower()

customers['Churn']=customers['Churn'].str.lower()
customers['Churn']=customers['Churn'].map({'yes':1,'no':0})
customers['Churn']=customers['Churn'].astype(int)
customers=customers.dropna(subset='Churn')

numeric=['tenure','MonthlyCharges','TotalCharges']
customers[numeric]=customers[numeric].apply(pd.to_numeric,errors='coerce').astype(float)
display(customers[numeric].isna().sum())

tenure             0
MonthlyCharges     0
TotalCharges      11
dtype: int64

In [18]:
train , test =train_test_split(
    customers,
    test_size=0.2,
    random_state=42,
    stratify=customers['Churn'])
train=train.copy()
test=test.copy()

In [22]:
medians=train[numeric].median()

categorical=train.select_dtypes(include=['object','string']).columns

train[numeric]=train[numeric].fillna(medians)
train[categorical]=train[categorical].fillna('Unknown')

test[numeric]=test[numeric].fillna(medians)
test[categorical]=test[categorical].fillna('Unknown')

In [24]:
for column in ['MonthlyCharges', 'TotalCharges']:

    q1 = train[column].quantile(0.25)
    q3 = train[column].quantile(0.75)

    iqr = q3 - q1

    lower = max(0, q1 - 1.5 * iqr)
    upper = q3 + 1.5 * iqr

    # Limit extreme values
    train[column] = train[column].clip(lower, upper)
    test[column] = test[column].clip(lower, upper)

In [26]:
# New customer: tenure less than 12 months
train['NewCustomer'] = (train['tenure'] < 12).astype(int)
test['NewCustomer'] = (test['tenure'] < 12).astype(int)

In [28]:
cleaned = pd.concat([train, test]).sort_index()

display(cleaned.head())
display(cleaned.isna().sum())

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,NewCustomer
0,7590-VHVEG,female,0,Yes,No,1.0,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,1
1,5575-GNVDE,male,0,No,No,34.0,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,0,0
2,3668-QPYBK,male,0,No,No,2.0,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,1
3,7795-CFOCW,male,0,No,No,45.0,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0,0
4,9237-HQITU,female,0,No,No,2.0,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,1


customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
NewCustomer         0
dtype: int64

In [30]:
X_train = pd.get_dummies(
    train.drop(columns=['customerID', 'Churn']),
    dtype=int
)

X_test = pd.get_dummies(
    test.drop(columns=['customerID', 'Churn']),
    dtype=int
)

In [32]:
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (5634, 46)
Testing shape: (1409, 46)


In [38]:
scaled_columns = numeric

scaler = StandardScaler()

X_train[scaled_columns] = scaler.fit_transform(
    X_train[scaled_columns]
)

X_test[scaled_columns] = scaler.transform(
    X_test[scaled_columns]
)

In [42]:

cleaned.to_csv(
    '06_cleaned_telecom.csv',
    index=False
)

X_train.assign(
    Churn=train['Churn']
).to_csv(
    '06_train.csv',
    index=False
)

X_test.assign(
    Churn=test['Churn']
).to_csv(
    '06_test.csv',
    index=False
)